# Random forest — lbw cohort

Train **random forest** on the `lbw` cohort and tune it with cluster-grouped
cross-validation on the training rows, then score the held-out test split once (section 4).

Owner: _(Jeremy Mwangi)_  ·  **`random_state=42` so results reproduce.**

## 1 — Load the data

In [ ]:
import sys
from pathlib import Path

# make the repo root importable when run from ml/notebooks/modeling
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml" / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.src.dataset import build_cohort, cv_folds, SELECTED, OUTCOMES

COHORT = "lbw"
# load the combined, cleaned cohort (needs both ke_lbw.csv and tz_lbw.csv in data/processed)
df = build_cohort(COHORT)

# training rows for tuning; test rows are used once, in section 4
train = df[df["split"] == "train"].reset_index(drop=True)
print(f"train rows: {len(train):,}  clusters: {train['cluster_id'].nunique():,}")

## 2 — Features and target
Features are the cohort's `SELECTED` columns minus the outcome and the design columns
(same rule as the other `lbw` notebooks, so the models are comparable).

In [ ]:
TARGET = OUTCOMES[COHORT]
DESIGN = ["split", "cluster_id", "country", "weight"]

# features = SELECTED minus outcome and design columns
FEATURES = [c for c in SELECTED[COHORT] if c not in DESIGN + [TARGET]]
X_train, y_train = train[FEATURES], train[TARGET]

# event count and rate (the constant-baseline PR-AUC)
print(f"{len(FEATURES)} features: {FEATURES}")
print(f"events: {int(y_train.sum()):,}  rate: {y_train.mean():.2%}")
# blanks per feature (first pregnancies have no interval by design)
print(X_train.isna().sum()[lambda s: s > 0].to_dict())

## 3 — Train and tune
`RandomForestClassifier` with `class_weight='balanced'` for the rare outcome (no resampling).
scikit-learn forests need complete inputs here, so the pipeline fills the blanks explicitly:
a first pregnancy's missing interval becomes -1 (the `first_pregnancy` column already marks it),
and the few blank ANC items get the training median. Imputers are fitted inside each CV fold.
`min_samples_leaf` and `max_depth` are the main guards against overfitting at this sample size.
Tuned with `GridSearchCV` on `cv_folds()` (5 cluster-grouped folds), PR-AUC first.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

# education is ordered, so encode it as 0-3 (the trees can split on order)
EDU_ORDER = ["No education", "Primary", "Secondary", "Higher"]
ANC_ITEMS = [c for c in ["no_anc", "anc4plus"] if c in FEATURES]
REST = [c for c in FEATURES if c not in ["education", "preceding_interval_m"] + ANC_ITEMS]

prep = ColumnTransformer(
    [
        ("edu", OrdinalEncoder(categories=[EDU_ORDER]), ["education"]),
        ("interval", SimpleImputer(strategy="constant", fill_value=-1), ["preceding_interval_m"]),
        ("anc", SimpleImputer(strategy="median"), ANC_ITEMS),
        ("rest", "passthrough", REST),
    ],
    verbose_feature_names_out=False,
)
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
pipe = Pipeline([("prep", prep), ("model", rf)])

# small grid of simple forests: 18 settings x 5 folds
GRID = {
    "model__min_samples_leaf": [10, 30, 80],
    "model__max_depth": [5, 8, None],
    "model__max_features": ["sqrt", 0.5],
}

# cluster-grouped folds from dataset.py
folds = cv_folds(train, COHORT)
search = GridSearchCV(pipe, GRID, cv=folds, scoring=["average_precision", "roc_auc"],
                      refit="average_precision", n_jobs=1)
# fit on training rows only
search.fit(X_train, y_train)

In [ ]:
# per-fold scores of the best setting
res, k = search.cv_results_, search.best_index_
fold_scores = pd.DataFrame({
    "pr_auc": [res[f"split{i}_test_average_precision"][k] for i in range(len(folds))],
    "roc_auc": [res[f"split{i}_test_roc_auc"][k] for i in range(len(folds))],
    # constant baseline: PR-AUC of a flat score = event rate in the fold
    "baseline_pr_auc": [y_train.iloc[val].mean() for _, val in folds],
}, index=pd.Index(range(1, len(folds) + 1), name="fold"))

# mean and sd over folds
summary = fold_scores.agg(["mean", "std"]).T.round(4)
print(summary)
print("best params:", {p.replace("model__", ""): v for p, v in search.best_params_.items()})

### CV results (training rows only)

- PR-AUC **0.198 ± 0.019** over 5 cluster-grouped folds, against a constant baseline of 0.082 (about 2.4×).
- ROC-AUC **0.618 ± 0.029**.
- Best setting: max_depth 5, max_features sqrt, min_samples_leaf 10 (300 trees).
- Two of the three best values sit at the edge of the grid (max_depth 5 and min_samples_leaf 10 are the lowest values searched), so a wider grid could move the result; the scores are close enough that I would not expect a large change.

In [ ]:
# fold scores against the constant baseline
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(fold_scores.index, fold_scores["pr_auc"], label="random forest")
axes[0].plot(fold_scores.index, fold_scores["baseline_pr_auc"], "k--o", label="constant baseline")
axes[0].set(title="CV PR-AUC by fold", xlabel="fold", ylabel="PR-AUC")
axes[0].legend()
axes[1].bar(fold_scores.index, fold_scores["roc_auc"])
axes[1].axhline(0.5, color="k", ls="--")
axes[1].set(title="CV ROC-AUC by fold", xlabel="fold", ylabel="ROC-AUC", ylim=(0.4, 1))
fig.suptitle(f"{COHORT}: random forest, best grid setting")
fig.tight_layout()
plt.show()

## 4 — Evaluate on the test split

The best CV setting, already refit on all training rows by `GridSearchCV`, scores the held-out
test clusters once. Nothing is retuned after this. PR-AUC is the headline; the checklist comparison
flags the same share of test pregnancies as the current high-risk checklist.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

# test rows: the held-out clusters, used once
test = df[df["split"] == "test"].reset_index(drop=True)
X_test, y_test = test[FEATURES], test[TARGET]
print(f"test rows: {len(test):,}  events: {int(y_test.sum()):,}  rate: {y_test.mean():.2%}")

# risk scores from the best setting (refit on all training rows)
model = search.best_estimator_
score = model.predict_proba(X_test)[:, 1]

In [ ]:
# test PR-AUC and ROC-AUC, pooled and per country (country is not a feature)
def metrics(mask):
    y, s = y_test[mask], score[mask]
    return {"rows": int(mask.sum()), "events": int(y.sum()), "baseline_pr_auc": y.mean(),
            "pr_auc": average_precision_score(y, s), "roc_auc": roc_auc_score(y, s)}


test_scores = pd.DataFrame({
    "pooled": metrics(np.ones(len(test), bool)),
    "Kenya": metrics((test["country"] == "Kenya").to_numpy()),
    "Tanzania": metrics((test["country"] == "Tanzania").to_numpy()),
}).T
print(test_scores.round(3))

# CV figures for comparison
print(summary.loc[["pr_auc", "roc_auc"]])

In [ ]:
# checklist flag: same rule as checklist_flag in dataset.py (not in SELECTED, so rebuilt here)
checklist = (
    (test["maternal_age"] < 18) | (test["maternal_age"] >= 35)
    | (test["prior_live_births"] >= 5) | (test["preceding_interval_m"] < 24)
    | (test["multiple"] == 1) | (test["prior_stillbirth"] == 1)
).to_numpy()
n_flag = int(checklist.sum())

# model flags the same number of pregnancies: the highest scores
model_flag = np.zeros(len(test), bool)
model_flag[np.argsort(-score, kind="stable")[:n_flag]] = True

# events caught, sensitivity and PPV at the same flag rate
events = y_test.to_numpy() == 1
compare = pd.DataFrame({
    "flagged": [n_flag, n_flag],
    "events_caught": [int((checklist & events).sum()), int((model_flag & events).sum())],
}, index=["checklist", "random forest"])
compare["sensitivity"] = compare["events_caught"] / events.sum()
compare["ppv"] = compare["events_caught"] / compare["flagged"]
print(f"flag rate: {checklist.mean():.1%} of test pregnancies ({n_flag:,})")
print(compare.round(3))

# many rows share a score at the cut-off: range of events the model could catch
cut = np.sort(score)[::-1][n_flag - 1]
above, tied = score > cut, score == cut
slots = n_flag - int(above.sum())
low = int((above & events).sum()) + max(0, slots - int((tied & ~events).sum()))
high = int((above & events).sum()) + min(slots, int((tied & events).sum()))
print(f"{int(tied.sum())} rows tied at the cut-off; model catches {low}-{high} events")

In [ ]:
# how sure are we? bootstrap over clusters (rows in a cluster are not independent)
rng = np.random.default_rng(42)
cl = test["cluster_id"].to_numpy()
by_cl = {k: np.flatnonzero(cl == k) for k in np.unique(cl)}
keys = np.array(list(by_cl))
boot = []
for _ in range(300):
    pick = np.concatenate([by_cl[k] for k in rng.choice(keys, len(keys))])
    if y_test.iloc[pick].nunique() == 2:
        boot.append((average_precision_score(y_test.iloc[pick], score[pick]),
                     roc_auc_score(y_test.iloc[pick], score[pick])))
ci_lo, ci_hi = np.percentile(np.array(boot), [2.5, 97.5], axis=0)   # each: [PR-AUC, ROC-AUC]
print(f"PR-AUC 95% CI {ci_lo[0]:.3f}-{ci_hi[0]:.3f}   ROC-AUC 95% CI {ci_lo[1]:.3f}-{ci_hi[1]:.3f}")

In [ ]:
# calibration: mean predicted vs observed, and 10 equal-width bins
print(f"mean predicted {score.mean():.3f}  observed rate {y_test.mean():.3f}")
obs, pred = calibration_curve(y_test, score, n_bins=10)

# PR curve with the constant baseline and the checklist point; reliability plot
prec, rec, _ = precision_recall_curve(y_test, score)
pooled_pr = test_scores.loc["pooled", "pr_auc"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(rec, prec, label=f"random forest (PR-AUC {pooled_pr:.3f})")
axes[0].axhline(y_test.mean(), color="k", ls="--", label=f"constant baseline ({y_test.mean():.3f})")
axes[0].plot(compare.loc["checklist", "sensitivity"], compare.loc["checklist", "ppv"], "rs",
             label="checklist")
axes[0].set(title="Test precision-recall", xlabel="recall (sensitivity)", ylabel="precision (PPV)")
axes[0].legend()
axes[1].plot(pred, obs, "o-", label="random forest")
axes[1].plot([0, 1], [0, 1], "k--", label="perfect calibration")
axes[1].set(title="Test reliability (10 bins)", xlabel="mean predicted", ylabel="observed rate")
axes[1].legend()
fig.suptitle(f"{COHORT}: random forest on the test split")
fig.tight_layout()
plt.show()

In [ ]:
# which inputs matter? permutation importance on the test rows (drop in PR-AUC).
# Quick look only; the SHAP drivers for the winning model come in validation_shap.ipynb.
from sklearn.inspection import permutation_importance

imp = permutation_importance(model, X_test, y_test, scoring="average_precision",
                             n_repeats=20, random_state=42, n_jobs=-1)
order = np.argsort(imp.importances_mean)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(np.array(FEATURES)[order], imp.importances_mean[order], xerr=imp.importances_std[order])
ax.set(title="Permutation importance (drop in test PR-AUC)")
fig.tight_layout()
plt.show()

### Test results

- **PR-AUC 0.137** on 1,849 test births (130 LBW), against a constant baseline of 0.070: about 2.0× better than flagging at random. Bootstrap over clusters (300 resamples): 95% CI 0.103–0.195. ROC-AUC 0.626 (0.573–0.676).
- Lower than CV (0.198), but the test event rate is also lower (7.0% vs 8.2% in training). The gain over the baseline is 2.0× vs 2.4× in CV, and ROC-AUC held (0.626 vs 0.618 ± 0.029).
- Tanzania is stronger than Kenya: PR-AUC 0.221 vs 0.107 (baselines 0.067 and 0.074), ROC-AUC 0.628 vs 0.618.
- The checklist flags 41.6% of test births (770) and catches 69 of 130 LBW (sensitivity 0.531, PPV 0.090).
- Flagging the same 770 by random forest score catches 79 (sensitivity 0.608, PPV 0.103): 10 more events, with no ties at the cut-off (79–79). One split with 130 events is suggestive, not conclusive.
- Probabilities are not risks: mean predicted 0.468 against an observed rate of 0.070, because `class_weight="balanced"` inflates them. Use the score to rank only.
- For comparison, the gradient boosting notebook reports test PR-AUC 0.159 and 73 events caught at the same flag rate on the same test split. The random forest has the lower PR-AUC but catches 6 more events at the checklist's flag rate; both differences are well inside the bootstrap CI, so neither model is clearly better. Compare with logistic regression before choosing.

## 5 — Save the result

Values for `db.metrics.save_metrics(...)` (one row in `model_metrics`). `recall_at_flag` is the
model's sensitivity at the checklist's flag rate. Set `SAVE_TO_DB = True` once the app Postgres is up.

In [ ]:
row = {
    "cohort": COHORT,
    "algorithm": "random_forest",
    "pr_auc": float(test_scores.loc["pooled", "pr_auc"]),
    "roc_auc": float(test_scores.loc["pooled", "roc_auc"]),
    "recall_at_flag": float(compare.loc["random forest", "sensitivity"]),
    "beats_checklist": bool(compare.loc["random forest", "events_caught"]
                            > compare.loc["checklist", "events_caught"]),
    "test_events": int(y_test.sum()),
}
print(pd.DataFrame([row]).T)

SAVE_TO_DB = False   # needs the app DB (docker compose up -d app_db)
if SAVE_TO_DB:
    from db.metrics import save_metrics
    save_metrics(**row)

SAVE_MODEL = False   # only if this is the cohort winner; ml/models/ is git-ignored
if SAVE_MODEL:
    import joblib
    (ROOT / "ml" / "models").mkdir(exist_ok=True)
    joblib.dump(search.best_estimator_, ROOT / "ml" / "models" / "lbw_random_forest.joblib")